# Pertemuan 2: Python untuk Big Data — Polars Deep Dive

## Tujuan Pembelajaran
1. Memahami arsitektur Polars: Rust, Apache Arrow, dan eksekusi multi-thread
2. Membedakan dan memilih antara Lazy vs Eager API
3. Menguasai Polars Expressions untuk manipulasi data kolumnar
4. Mengoptimalkan pipeline dengan chaining, predicate pushdown, dan projection pushdown
5. Menulis data ke format Parquet yang efisien



---
## Bagian 1: Setup dan Pembuatan Data Simulasi

Kita akan membuat dataset e-commerce simulasi Indonesia dengan 2 juta baris
untuk mendemonstrasikan keunggulan Polars pada data berskala besar.



In [22]:
import polars as pl
import numpy as np
import time
import os

# Seed untuk reproduktibilitas
np.random.seed(2026)

N = 2_000_000
print(f"Membuat dataset simulasi {N:,} baris...")

kota_list = [
    "Jakarta", "Surabaya", "Bandung", "Medan", "Semarang",
    "Makassar", "Palembang", "Tangerang", "Depok", "Bogor",
    "Yogyakarta", "Bekasi", "Denpasar", "Balikpapan", "Manado",
]
kategori_list = ["Elektronik", "Fashion", "Makanan", "Kesehatan", "Olahraga", "Kecantikan", "Otomotif"]
status_list = ["selesai", "dikirim", "proses", "dibatalkan", "dikembalikan"]

data = {
    "transaksi_id": [f"TRX{i:08d}" for i in range(N)],
    "tanggal": pl.datetime_range(
        pl.datetime(2024, 1, 1), pl.datetime(2024, 12, 31), interval="1s",
        eager=True
    ).sample(N, with_replacement=True, seed=2026).alias("tanggal"),
    "kota_penjual": np.random.choice(kota_list, N),
    "kota_pembeli": np.random.choice(kota_list, N),
    "kategori": np.random.choice(kategori_list, N),
    "harga": np.random.randint(5_000, 10_000_000, N),
    "kuantitas": np.random.randint(1, 20, N),
    "diskon_persen": np.random.choice([0, 0, 0, 5, 10, 15, 20, 25, 30, 50], N),
    "rating": np.round(np.random.uniform(1.0, 5.0, N), 1),
    "status": np.random.choice(status_list, N, p=[0.65, 0.15, 0.10, 0.06, 0.04]),
}

# Buat DataFrame Polars
df = pl.DataFrame(data)
print(f"Dataset berhasil dibuat: {df.shape[0]:,} baris x {df.shape[1]} kolom")
print(df.head(5))



Membuat dataset simulasi 2,000,000 baris...
Dataset berhasil dibuat: 2,000,000 baris x 10 kolom
shape: (5, 10)
┌────────────┬────────────┬────────────┬────────────┬───┬───────────┬───────────┬────────┬─────────┐
│ transaksi_ ┆ tanggal    ┆ kota_penju ┆ kota_pembe ┆ … ┆ kuantitas ┆ diskon_pe ┆ rating ┆ status  │
│ id         ┆ ---        ┆ al         ┆ li         ┆   ┆ ---       ┆ rsen      ┆ ---    ┆ ---     │
│ ---        ┆ datetime[μ ┆ ---        ┆ ---        ┆   ┆ i32       ┆ ---       ┆ f64    ┆ str     │
│ str        ┆ s]         ┆ str        ┆ str        ┆   ┆           ┆ i64       ┆        ┆         │
╞════════════╪════════════╪════════════╪════════════╪═══╪═══════════╪═══════════╪════════╪═════════╡
│ TRX0000000 ┆ 2024-06-04 ┆ Surabaya   ┆ Surabaya   ┆ … ┆ 19        ┆ 0         ┆ 3.1    ┆ selesai │
│ 0          ┆ 20:31:44   ┆            ┆            ┆   ┆           ┆           ┆        ┆         │
│ TRX0000000 ┆ 2024-08-12 ┆ Palembang  ┆ Makassar   ┆ … ┆ 9         ┆ 10        ┆

---
## Bagian 2: Eager vs Lazy API

Polars memiliki dua mode eksekusi:
- **Eager**: Langsung dieksekusi saat perintah ditulis (seperti Pandas)
- **Lazy**: Membangun rencana eksekusi dulu, baru dijalankan saat `.collect()` dipanggil

Mode Lazy memungkinkan Polars mengoptimalkan query secara otomatis.



In [23]:
# --- EAGER API ---
# Eksekusi langsung, hasilnya langsung tersedia
print("=== Eager API ===")
start = time.perf_counter()
hasil_eager = (
    df
    .filter(pl.col("status") == "selesai")
    .group_by("kategori")
    .agg(
        pl.col("harga").mean().alias("rata_rata_harga"),
        pl.col("kuantitas").sum().alias("total_kuantitas"),
    )
    .sort("rata_rata_harga", descending=True)
)
waktu_eager = time.perf_counter() - start
print(f"Waktu Eager: {waktu_eager:.4f} detik")
print(hasil_eager)



=== Eager API ===
Waktu Eager: 0.0423 detik
shape: (7, 3)
┌────────────┬─────────────────┬─────────────────┐
│ kategori   ┆ rata_rata_harga ┆ total_kuantitas │
│ ---        ┆ ---             ┆ ---             │
│ str        ┆ f64             ┆ i32             │
╞════════════╪═════════════════╪═════════════════╡
│ Kecantikan ┆ 5.0142e6        ┆ 1855952         │
│ Elektronik ┆ 5.0097e6        ┆ 1863058         │
│ Kesehatan  ┆ 4.9983e6        ┆ 1863700         │
│ Makanan    ┆ 4.9961e6        ┆ 1857576         │
│ Otomotif   ┆ 4.9952e6        ┆ 1854180         │
│ Fashion    ┆ 4.9906e6        ┆ 1861318         │
│ Olahraga   ┆ 4.9898e6        ┆ 1854904         │
└────────────┴─────────────────┴─────────────────┘


In [24]:
# --- LAZY API ---
# Membangun rencana eksekusi (LazyFrame), baru dieksekusi saat .collect()
print("\n=== Lazy API ===")
start = time.perf_counter()
hasil_lazy = (
    df.lazy()                                     # Ubah ke LazyFrame
    .filter(pl.col("status") == "selesai")        # Filter dulu (predicate pushdown)
    .group_by("kategori")
    .agg(
        pl.col("harga").mean().alias("rata_rata_harga"),
        pl.col("kuantitas").sum().alias("total_kuantitas"),
    )
    .sort("rata_rata_harga", descending=True)
    .collect()                                    # Baru dieksekusi di sini
)
waktu_lazy = time.perf_counter() - start
print(f"Waktu Lazy:  {waktu_lazy:.4f} detik")
print(hasil_lazy)




=== Lazy API ===
Waktu Lazy:  0.0213 detik
shape: (7, 3)
┌────────────┬─────────────────┬─────────────────┐
│ kategori   ┆ rata_rata_harga ┆ total_kuantitas │
│ ---        ┆ ---             ┆ ---             │
│ str        ┆ f64             ┆ i32             │
╞════════════╪═════════════════╪═════════════════╡
│ Kecantikan ┆ 5.0142e6        ┆ 1855952         │
│ Elektronik ┆ 5.0097e6        ┆ 1863058         │
│ Kesehatan  ┆ 4.9983e6        ┆ 1863700         │
│ Makanan    ┆ 4.9961e6        ┆ 1857576         │
│ Otomotif   ┆ 4.9952e6        ┆ 1854180         │
│ Fashion    ┆ 4.9906e6        ┆ 1861318         │
│ Olahraga   ┆ 4.9898e6        ┆ 1854904         │
└────────────┴─────────────────┴─────────────────┘


In [25]:
# Melihat rencana eksekusi Lazy (query plan)
print("\n=== Query Plan (Lazy) ===")
q = (
    df.lazy()
    .filter(pl.col("status") == "selesai")
    .group_by("kategori")
    .agg(pl.col("harga").mean().alias("rata_rata_harga"))
    .sort("rata_rata_harga", descending=True)
)
# Tampilkan rencana query yang dioptimalkan
print(q.explain(optimized=True))




=== Query Plan (Lazy) ===
SORT BY [descending: [true]] [col("rata_rata_harga")]
  AGGREGATE[maintain_order: false]
    [col("harga").mean().alias("rata_rata_harga")] BY [col("kategori")]
    FROM
    simple π 2/2 ["kategori", "harga"]
      FILTER col("status") == "selesai"
      FROM
        DF ["transaksi_id", "tanggal", "kota_penjual", "kota_pembeli", ...]; PROJECT["kategori", "harga", "status"] 3/10 COLUMNS


---
## Bagian 3: Polars Expressions

Expressions adalah inti dari Polars. Berbeda dengan Pandas yang bekerja baris per baris,
Polars bekerja kolom per kolom secara paralel menggunakan Apache Arrow.

### 3a. Ekspresi Dasar: select dan with_columns



In [26]:
# select: memilih dan mentransformasi kolom
print("=== select: memilih kolom ===")
hasil = df.select([
    pl.col("transaksi_id"),
    pl.col("kategori"),
    pl.col("harga"),
    pl.col("kuantitas"),
    # Membuat kolom baru: total = harga * kuantitas * (1 - diskon/100)
    (pl.col("harga") * pl.col("kuantitas") * (1 - pl.col("diskon_persen") / 100))
    .alias("total_bayar"),
])
print(hasil.head(5))



=== select: memilih kolom ===
shape: (5, 5)
┌──────────────┬────────────┬─────────┬───────────┬─────────────┐
│ transaksi_id ┆ kategori   ┆ harga   ┆ kuantitas ┆ total_bayar │
│ ---          ┆ ---        ┆ ---     ┆ ---       ┆ ---         │
│ str          ┆ str        ┆ i32     ┆ i32       ┆ f64         │
╞══════════════╪════════════╪═════════╪═══════════╪═════════════╡
│ TRX00000000  ┆ Kesehatan  ┆ 4453749 ┆ 19        ┆ 8.4621231e7 │
│ TRX00000001  ┆ Olahraga   ┆ 6270594 ┆ 9         ┆ 5.0792e7    │
│ TRX00000002  ┆ Fashion    ┆ 3809756 ┆ 3         ┆ 5.714634e6  │
│ TRX00000003  ┆ Kecantikan ┆ 6025305 ┆ 7         ┆ 3.3741708e7 │
│ TRX00000004  ┆ Kesehatan  ┆ 7651505 ┆ 11        ┆ 8.4166555e7 │
└──────────────┴────────────┴─────────┴───────────┴─────────────┘


In [27]:
# with_columns: menambahkan kolom baru tanpa menghapus kolom lama
print("\n=== with_columns: menambahkan kolom ===")
df_enriched = df.with_columns([
    # Total bayar setelah diskon
    (pl.col("harga") * pl.col("kuantitas") * (1 - pl.col("diskon_persen") / 100))
    .cast(pl.Int64)
    .alias("total_bayar"),
    # Klasifikasi harga
    pl.when(pl.col("harga") < 100_000).then(pl.lit("Murah"))
    .when(pl.col("harga") < 1_000_000).then(pl.lit("Menengah"))
    .otherwise(pl.lit("Premium"))
    .alias("segmen_harga"),
    # Apakah transaksi berhasil?
    (pl.col("status") == "selesai").alias("berhasil"),
])
print(df_enriched.select(["transaksi_id", "harga", "total_bayar", "segmen_harga", "berhasil"]).head(8))




=== with_columns: menambahkan kolom ===
shape: (8, 5)
┌──────────────┬─────────┬─────────────┬──────────────┬──────────┐
│ transaksi_id ┆ harga   ┆ total_bayar ┆ segmen_harga ┆ berhasil │
│ ---          ┆ ---     ┆ ---         ┆ ---          ┆ ---      │
│ str          ┆ i32     ┆ i64         ┆ str          ┆ bool     │
╞══════════════╪═════════╪═════════════╪══════════════╪══════════╡
│ TRX00000000  ┆ 4453749 ┆ 84621231    ┆ Premium      ┆ true     │
│ TRX00000001  ┆ 6270594 ┆ 50791811    ┆ Premium      ┆ true     │
│ TRX00000002  ┆ 3809756 ┆ 5714634     ┆ Premium      ┆ true     │
│ TRX00000003  ┆ 6025305 ┆ 33741708    ┆ Premium      ┆ true     │
│ TRX00000004  ┆ 7651505 ┆ 84166555    ┆ Premium      ┆ true     │
│ TRX00000005  ┆ 7947732 ┆ 47686392    ┆ Premium      ┆ true     │
│ TRX00000006  ┆ 1579460 ┆ 4738380     ┆ Premium      ┆ true     │
│ TRX00000007  ┆ 8635007 ┆ 102756583   ┆ Premium      ┆ true     │
└──────────────┴─────────┴─────────────┴──────────────┴──────────┘


### 3b. Filter: Kondisi dan Logika Boolean



In [28]:
# Filter dengan berbagai kondisi
print("=== Filter multi-kondisi ===")

# Transaksi besar yang selesai di Jakarta atau Surabaya
transaksi_besar = (
    df
    .filter(
        (pl.col("status") == "selesai") &
        (pl.col("harga") > 1_000_000) &
        (pl.col("kota_penjual").is_in(["Jakarta", "Surabaya"]))
    )
)
print(f"Jumlah transaksi besar: {len(transaksi_besar):,}")
print(transaksi_besar.head(5))



=== Filter multi-kondisi ===
Jumlah transaksi besar: 156,297
shape: (5, 10)
┌────────────┬────────────┬────────────┬────────────┬───┬───────────┬───────────┬────────┬─────────┐
│ transaksi_ ┆ tanggal    ┆ kota_penju ┆ kota_pembe ┆ … ┆ kuantitas ┆ diskon_pe ┆ rating ┆ status  │
│ id         ┆ ---        ┆ al         ┆ li         ┆   ┆ ---       ┆ rsen      ┆ ---    ┆ ---     │
│ ---        ┆ datetime[μ ┆ ---        ┆ ---        ┆   ┆ i32       ┆ ---       ┆ f64    ┆ str     │
│ str        ┆ s]         ┆ str        ┆ str        ┆   ┆           ┆ i64       ┆        ┆         │
╞════════════╪════════════╪════════════╪════════════╪═══╪═══════════╪═══════════╪════════╪═════════╡
│ TRX0000000 ┆ 2024-06-04 ┆ Surabaya   ┆ Surabaya   ┆ … ┆ 19        ┆ 0         ┆ 3.1    ┆ selesai │
│ 0          ┆ 20:31:44   ┆            ┆            ┆   ┆           ┆           ┆        ┆         │
│ TRX0000001 ┆ 2024-07-13 ┆ Jakarta    ┆ Yogyakarta ┆ … ┆ 10        ┆ 0         ┆ 3.8    ┆ selesai │
│ 7          ┆ 

In [29]:
# Filter dengan ekspresi string
print("\n=== Filter dengan string operations ===")
transaksi_elektronik = df.filter(
    pl.col("kategori").str.to_lowercase().str.contains("elekt")
)
print(f"Transaksi elektronik: {len(transaksi_elektronik):,}")




=== Filter dengan string operations ===
Transaksi elektronik: 286,407


### 3c. group_by dan Agregasi



In [30]:
# Agregasi lengkap per kategori
print("=== Agregasi per kategori ===")
ringkasan_kategori = (
    df_enriched
    .group_by("kategori")
    .agg([
        pl.col("transaksi_id").count().alias("jumlah_transaksi"),
        pl.col("total_bayar").sum().alias("total_pendapatan"),
        pl.col("total_bayar").mean().alias("rata_rata_order"),
        pl.col("rating").mean().round(2).alias("rata_rata_rating"),
        pl.col("berhasil").sum().alias("transaksi_selesai"),
        pl.col("diskon_persen").mean().round(1).alias("rata_rata_diskon"),
    ])
    .with_columns(
        (pl.col("transaksi_selesai") / pl.col("jumlah_transaksi") * 100)
        .round(1)
        .alias("persen_berhasil")
    )
    .sort("total_pendapatan", descending=True)
)
print(ringkasan_kategori)



=== Agregasi per kategori ===
shape: (7, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ kategori   ┆ jumlah_tra ┆ total_pend ┆ rata_rata ┆ rata_rata ┆ transaksi ┆ rata_rata ┆ persen_be │
│ ---        ┆ nsaksi     ┆ apatan     ┆ _order    ┆ _rating   ┆ _selesai  ┆ _diskon   ┆ rhasil    │
│ str        ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆ u32        ┆ i64        ┆ f64       ┆ f64       ┆ u32       ┆ f64       ┆ f64       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Elektronik ┆ 286407     ┆ 1210670201 ┆ 4.2271e7  ┆ 3.0       ┆ 186200    ┆ 15.5      ┆ 65.0      │
│            ┆            ┆ 0287       ┆           ┆           ┆           ┆           ┆           │
│ Kesehatan  ┆ 286067     ┆ 1209803653 ┆ 4.2291e7  ┆ 3.0       ┆ 186142    ┆ 15.5      ┆ 65.1      │
│            ┆            ┆ 9096       ┆       

In [31]:
# Agregasi per kota dan kategori (multi-key group_by)
print("\n=== Top 5 kombinasi kota-kategori ===")
top_kombinasi = (
    df_enriched
    .filter(pl.col("status") == "selesai")
    .group_by(["kota_penjual", "kategori"])
    .agg(
        pl.col("total_bayar").sum().alias("total_pendapatan"),
        pl.col("transaksi_id").count().alias("jumlah"),
    )
    .sort("total_pendapatan", descending=True)
    .head(10)
)
print(top_kombinasi)




=== Top 5 kombinasi kota-kategori ===
shape: (10, 4)
┌──────────────┬────────────┬──────────────────┬────────┐
│ kota_penjual ┆ kategori   ┆ total_pendapatan ┆ jumlah │
│ ---          ┆ ---        ┆ ---              ┆ ---    │
│ str          ┆ str        ┆ i64              ┆ u32    │
╞══════════════╪════════════╪══════════════════╪════════╡
│ Bogor        ┆ Kesehatan  ┆ 543529627999     ┆ 12780  │
│ Semarang     ┆ Kecantikan ┆ 535628560128     ┆ 12656  │
│ Jakarta      ┆ Otomotif   ┆ 534255869053     ┆ 12523  │
│ Denpasar     ┆ Elektronik ┆ 534025612542     ┆ 12559  │
│ Tangerang    ┆ Fashion    ┆ 533614760565     ┆ 12492  │
│ Jakarta      ┆ Makanan    ┆ 532265342716     ┆ 12564  │
│ Jakarta      ┆ Elektronik ┆ 532168904991     ┆ 12497  │
│ Depok        ┆ Kesehatan  ┆ 531513355148     ┆ 12418  │
│ Semarang     ┆ Kesehatan  ┆ 531449966411     ┆ 12565  │
│ Tangerang    ┆ Kesehatan  ┆ 531437852414     ┆ 12466  │
└──────────────┴────────────┴──────────────────┴────────┘


---
## Bagian 4: Lazy Loading dengan scan_parquet dan scan_csv

`scan_csv` dan `scan_parquet` adalah versi lazy dari `read_csv`/`read_parquet`.
Data belum dibaca ke memori sampai `.collect()` dipanggil.
Ini sangat efisien untuk file berukuran sangat besar.



In [32]:
# Simpan dataset ke file terlebih dahulu
print("Menyimpan data ke file...")
os.makedirs("/tmp/bigdata_demo", exist_ok=True)
parquet_path = "/tmp/bigdata_demo/transaksi.parquet"
csv_path = "/tmp/bigdata_demo/transaksi.csv"

df.write_parquet(parquet_path)
df.write_csv(csv_path)
print(f"File tersimpan: {parquet_path}")
print(f"  Parquet: {os.path.getsize(parquet_path)/1e6:.1f} MB")
print(f"  CSV:     {os.path.getsize(csv_path)/1e6:.1f} MB")



Menyimpan data ke file...
File tersimpan: /tmp/bigdata_demo/transaksi.parquet
  Parquet: 28.4 MB
  CSV:     181.0 MB


In [33]:
# scan_parquet: lazy loading — hanya membaca kolom yang dibutuhkan
print("\n=== scan_parquet (Lazy) ===")
start = time.perf_counter()

hasil_scan = (
    pl.scan_parquet(parquet_path)      # Belum membaca file
    .filter(pl.col("status") == "selesai")   # Filter akan di-pushdown ke pembaca file
    .select(["kategori", "harga", "kuantitas"])  # Hanya baca 3 kolom (projection pushdown)
    .group_by("kategori")
    .agg(pl.col("harga").sum().alias("total"))
    .sort("total", descending=True)
    .collect()                                   # Baru baca file di sini
)
waktu_scan = time.perf_counter() - start
print(f"Waktu scan_parquet: {waktu_scan:.4f} detik")
print(hasil_scan)




=== scan_parquet (Lazy) ===
Waktu scan_parquet: 0.0233 detik
shape: (7, 2)
┌────────────┬─────────────┐
│ kategori   ┆ total       │
│ ---        ┆ ---         │
│ str        ┆ i32         │
╞════════════╪═════════════╡
│ Fashion    ┆ 1089198762  │
│ Elektronik ┆ 798315680   │
│ Olahraga   ┆ 733846920   │
│ Makanan    ┆ 244167143   │
│ Kecantikan ┆ -1018552965 │
│ Kesehatan  ┆ -1622824388 │
│ Otomotif   ┆ -1705862820 │
└────────────┴─────────────┘


In [34]:
# Perbandingan: read_parquet (eager) vs scan_parquet (lazy)
print("\n=== Perbandingan read_parquet vs scan_parquet ===")

# Eager: baca semua ke memori dulu
start = time.perf_counter()
_ = (
    pl.read_parquet(parquet_path)
    .filter(pl.col("status") == "selesai")
    .group_by("kategori")
    .agg(pl.col("harga").sum())
)
waktu_eager = time.perf_counter() - start

# Lazy: hanya baca kolom dan baris yang diperlukan
start = time.perf_counter()
_ = (
    pl.scan_parquet(parquet_path)
    .filter(pl.col("status") == "selesai")
    .select(["kategori", "harga"])
    .group_by("kategori")
    .agg(pl.col("harga").sum())
    .collect()
)
waktu_lazy = time.perf_counter() - start

print(f"read_parquet (eager): {waktu_eager:.4f} detik")
print(f"scan_parquet (lazy):  {waktu_lazy:.4f} detik")
print(f"Speedup lazy:         {waktu_eager/waktu_lazy:.1f}x lebih cepat")




=== Perbandingan read_parquet vs scan_parquet ===
read_parquet (eager): 0.0740 detik
scan_parquet (lazy):  0.0183 detik
Speedup lazy:         4.0x lebih cepat


---
## Bagian 5: Joins di Polars



In [35]:
# Buat tabel dimensi (master data kategori)
kategori_info = pl.DataFrame({
    "kategori": kategori_list,
    "divisi": ["Teknologi", "Gaya Hidup", "Konsumsi", "Kesehatan", "Gaya Hidup", "Kecantikan", "Otomotif"],
    "margin_persen": [12, 35, 8, 25, 30, 40, 15],
    "target_bulanan": [50_000_000_000, 30_000_000_000, 20_000_000_000,
                       15_000_000_000, 10_000_000_000, 8_000_000_000, 25_000_000_000],
})
print("=== Tabel Dimensi Kategori ===")
print(kategori_info)



=== Tabel Dimensi Kategori ===
shape: (7, 4)
┌────────────┬────────────┬───────────────┬────────────────┐
│ kategori   ┆ divisi     ┆ margin_persen ┆ target_bulanan │
│ ---        ┆ ---        ┆ ---           ┆ ---            │
│ str        ┆ str        ┆ i64           ┆ i64            │
╞════════════╪════════════╪═══════════════╪════════════════╡
│ Elektronik ┆ Teknologi  ┆ 12            ┆ 50000000000    │
│ Fashion    ┆ Gaya Hidup ┆ 35            ┆ 30000000000    │
│ Makanan    ┆ Konsumsi   ┆ 8             ┆ 20000000000    │
│ Kesehatan  ┆ Kesehatan  ┆ 25            ┆ 15000000000    │
│ Olahraga   ┆ Gaya Hidup ┆ 30            ┆ 10000000000    │
│ Kecantikan ┆ Kecantikan ┆ 40            ┆ 8000000000     │
│ Otomotif   ┆ Otomotif   ┆ 15            ┆ 25000000000    │
└────────────┴────────────┴───────────────┴────────────────┘


In [36]:
# Inner join: gabungkan transaksi dengan info kategori
print("\n=== Inner Join: Transaksi + Info Kategori ===")
df_joined = (
    df_enriched
    .join(kategori_info, on="kategori", how="inner")
    .with_columns(
        (pl.col("total_bayar") * pl.col("margin_persen") / 100)
        .alias("estimasi_profit")
    )
)
print(f"Jumlah baris setelah join: {len(df_joined):,}")
print(df_joined.select(["transaksi_id", "kategori", "divisi", "total_bayar", "estimasi_profit"]).head(5))




=== Inner Join: Transaksi + Info Kategori ===
Jumlah baris setelah join: 2,000,000
shape: (5, 5)
┌──────────────┬────────────┬────────────┬─────────────┬─────────────────┐
│ transaksi_id ┆ kategori   ┆ divisi     ┆ total_bayar ┆ estimasi_profit │
│ ---          ┆ ---        ┆ ---        ┆ ---         ┆ ---             │
│ str          ┆ str        ┆ str        ┆ i64         ┆ f64             │
╞══════════════╪════════════╪════════════╪═════════════╪═════════════════╡
│ TRX00000000  ┆ Kesehatan  ┆ Kesehatan  ┆ 84621231    ┆ 2.1155e7        │
│ TRX00000001  ┆ Olahraga   ┆ Gaya Hidup ┆ 50791811    ┆ 1.5238e7        │
│ TRX00000002  ┆ Fashion    ┆ Gaya Hidup ┆ 5714634     ┆ 2.0001e6        │
│ TRX00000003  ┆ Kecantikan ┆ Kecantikan ┆ 33741708    ┆ 1.3497e7        │
│ TRX00000004  ┆ Kesehatan  ┆ Kesehatan  ┆ 84166555    ┆ 2.1042e7        │
└──────────────┴────────────┴────────────┴─────────────┴─────────────────┘


In [37]:
# Analisis per divisi setelah join
print("\n=== Analisis per Divisi ===")
analisis_divisi = (
    df_joined
    .filter(pl.col("status") == "selesai")
    .group_by("divisi")
    .agg([
        pl.col("total_bayar").sum().alias("total_pendapatan"),
        pl.col("estimasi_profit").sum().alias("total_profit"),
        pl.col("transaksi_id").count().alias("jumlah_transaksi"),
    ])
    .sort("total_pendapatan", descending=True)
)
print(analisis_divisi)




=== Analisis per Divisi ===
shape: (6, 4)
┌────────────┬──────────────────┬──────────────┬──────────────────┐
│ divisi     ┆ total_pendapatan ┆ total_profit ┆ jumlah_transaksi │
│ ---        ┆ ---              ┆ ---          ┆ ---              │
│ str        ┆ i64              ┆ f64          ┆ u32              │
╞════════════╪══════════════════╪══════════════╪══════════════════╡
│ Gaya Hidup ┆ 15666932263445   ┆ 5.0927e12    ┆ 371318           │
│ Kesehatan  ┆ 7888066535413    ┆ 1.9720e12    ┆ 186142           │
│ Teknologi  ┆ 7885771163333    ┆ 9.4629e11    ┆ 186200           │
│ Kecantikan ┆ 7874474567466    ┆ 3.1498e12    ┆ 185671           │
│ Konsumsi   ┆ 7838325758977    ┆ 6.2707e11    ┆ 185738           │
│ Otomotif   ┆ 7818612027510    ┆ 1.1728e12    ┆ 185381           │
└────────────┴──────────────────┴──────────────┴──────────────────┘


---
## Bagian 6: Window Functions

Window functions memungkinkan agregasi dalam grup tanpa mereduksi jumlah baris.
Berguna untuk: ranking, running total, perbandingan dengan rata-rata grup, dll.



In [38]:
# Agregasi per kota untuk window function
print("=== Window Functions ===")

df_kota = (
    df_enriched
    .filter(pl.col("status") == "selesai")
    .group_by(["kota_penjual", "kategori"])
    .agg(pl.col("total_bayar").sum().alias("pendapatan"))
)

df_window = df_kota.with_columns([
    # Ranking pendapatan per kota (dalam grup kota_penjual)
    pl.col("pendapatan")
    .rank(method="dense", descending=True)
    .over("kota_penjual")
    .alias("rank_dalam_kota"),

    # Rata-rata pendapatan per kota (window aggregation)
    pl.col("pendapatan")
    .mean()
    .over("kota_penjual")
    .alias("rata_rata_kota"),

    # Persentase kontribusi terhadap total kota
    (pl.col("pendapatan") / pl.col("pendapatan").sum().over("kota_penjual") * 100)
    .round(1)
    .alias("persen_kontribusi"),
])

# Tampilkan top kategori per kota Jakarta
print(df_window.filter(pl.col("kota_penjual") == "Jakarta").sort("rank_dalam_kota"))



=== Window Functions ===
shape: (7, 6)
┌──────────────┬────────────┬──────────────┬─────────────────┬────────────────┬───────────────────┐
│ kota_penjual ┆ kategori   ┆ pendapatan   ┆ rank_dalam_kota ┆ rata_rata_kota ┆ persen_kontribusi │
│ ---          ┆ ---        ┆ ---          ┆ ---             ┆ ---            ┆ ---               │
│ str          ┆ str        ┆ i64          ┆ u32             ┆ f64            ┆ f64               │
╞══════════════╪════════════╪══════════════╪═════════════════╪════════════════╪═══════════════════╡
│ Jakarta      ┆ Otomotif   ┆ 534255869053 ┆ 1               ┆ 5.2852e11      ┆ 14.4              │
│ Jakarta      ┆ Makanan    ┆ 532265342716 ┆ 2               ┆ 5.2852e11      ┆ 14.4              │
│ Jakarta      ┆ Elektronik ┆ 532168904991 ┆ 3               ┆ 5.2852e11      ┆ 14.4              │
│ Jakarta      ┆ Kecantikan ┆ 529761664316 ┆ 4               ┆ 5.2852e11      ┆ 14.3              │
│ Jakarta      ┆ Olahraga   ┆ 526177210996 ┆ 5               

---
## Bagian 7: String Operations dan Temporal



In [39]:
# Operasi string dengan Polars
print("=== String Operations ===")
df_string = df.select([
    pl.col("transaksi_id"),
    pl.col("kota_penjual"),
    # Extract nomor dari ID transaksi
    pl.col("transaksi_id").str.replace("TRX", "").cast(pl.Int32).alias("nomor_urut"),
    # Gabungkan kota penjual dan pembeli
    (pl.col("kota_penjual") + " -> " + pl.col("kota_pembeli")).alias("rute"),
    # Apakah pengiriman dalam kota yang sama?
    (pl.col("kota_penjual") == pl.col("kota_pembeli")).alias("pengiriman_lokal"),
])
print(df_string.head(8))



=== String Operations ===
shape: (8, 5)
┌──────────────┬──────────────┬────────────┬──────────────────────────┬──────────────────┐
│ transaksi_id ┆ kota_penjual ┆ nomor_urut ┆ rute                     ┆ pengiriman_lokal │
│ ---          ┆ ---          ┆ ---        ┆ ---                      ┆ ---              │
│ str          ┆ str          ┆ i32        ┆ str                      ┆ bool             │
╞══════════════╪══════════════╪════════════╪══════════════════════════╪══════════════════╡
│ TRX00000000  ┆ Surabaya     ┆ 0          ┆ Surabaya -> Surabaya     ┆ true             │
│ TRX00000001  ┆ Palembang    ┆ 1          ┆ Palembang -> Makassar    ┆ false            │
│ TRX00000002  ┆ Yogyakarta   ┆ 2          ┆ Yogyakarta -> Medan      ┆ false            │
│ TRX00000003  ┆ Depok        ┆ 3          ┆ Depok -> Bandung         ┆ false            │
│ TRX00000004  ┆ Balikpapan   ┆ 4          ┆ Balikpapan -> Semarang   ┆ false            │
│ TRX00000005  ┆ Balikpapan   ┆ 5          ┆ Balik

In [40]:
# Operasi temporal (date/datetime)
print("\n=== Temporal Operations ===")
df_temporal = df.with_columns([
    pl.col("tanggal").dt.month().alias("bulan"),
    pl.col("tanggal").dt.quarter().alias("kuartal"),
    pl.col("tanggal").dt.weekday().alias("hari_dalam_minggu"),  # 0=Senin, 6=Minggu
]).select([
    "tanggal", "bulan", "kuartal", "hari_dalam_minggu",
    "kategori", "harga"
])

# Analisis penjualan per bulan
print("Penjualan per bulan:")
print(
    df_temporal
    .group_by("bulan")
    .agg(pl.col("harga").sum().alias("total_penjualan"))
    .sort("bulan")
)




=== Temporal Operations ===


Penjualan per bulan:
shape: (12, 2)
┌───────┬─────────────────┐
│ bulan ┆ total_penjualan │
│ ---   ┆ ---             │
│ i8    ┆ i32             │
╞═══════╪═════════════════╡
│ 1     ┆ -1849047097     │
│ 2     ┆ 1549631826      │
│ 3     ┆ -595555802      │
│ 4     ┆ -12746764       │
│ 5     ┆ 1487346764      │
│ …     ┆ …               │
│ 8     ┆ -753644922      │
│ 9     ┆ 1703301390      │
│ 10    ┆ -2065985942     │
│ 11    ┆ 2120583734      │
│ 12    ┆ -970799200      │
└───────┴─────────────────┘


---
## Bagian 8: Menulis Data ke Parquet dengan Partisi

Partisi Parquet memungkinkan query yang jauh lebih cepat karena hanya
folder/partisi yang relevan yang dibaca.



In [41]:
# Simpan dengan partisi per kategori
print("=== Menulis Parquet dengan Partisi ===")
output_dir = "/tmp/bigdata_demo/transaksi_partisi"

(
    df_enriched
    .with_columns(pl.col("tanggal").dt.month().alias("bulan"))
    .write_parquet(
        output_dir,
        use_pyarrow=True,
        pyarrow_options={"partition_cols": ["kategori", "bulan"]},
    )
)
print(f"Data terpartisi tersimpan di: {output_dir}")

# Lihat struktur folder partisi
for dirpath, dirnames, filenames in os.walk(output_dir):
    depth = dirpath.replace(output_dir, "").count(os.sep)
    if depth <= 2:
        indent = "  " * depth
        print(f"{indent}{os.path.basename(dirpath)}/")
        if filenames and depth == 2:
            for f in filenames[:1]:
                print(f"{indent}  {f}")



=== Menulis Parquet dengan Partisi ===
Data terpartisi tersimpan di: /tmp/bigdata_demo/transaksi_partisi
transaksi_partisi/
  kategori=Elektronik/
    bulan=1/
      18c7f654841d4098812d16a2d1767392-0.parquet
    bulan=10/
      18c7f654841d4098812d16a2d1767392-0.parquet
    bulan=11/
      18c7f654841d4098812d16a2d1767392-0.parquet
    bulan=12/
      18c7f654841d4098812d16a2d1767392-0.parquet
    bulan=2/
      18c7f654841d4098812d16a2d1767392-0.parquet
    bulan=3/
      18c7f654841d4098812d16a2d1767392-0.parquet
    bulan=4/
      18c7f654841d4098812d16a2d1767392-0.parquet
    bulan=5/
      18c7f654841d4098812d16a2d1767392-0.parquet
    bulan=6/
      18c7f654841d4098812d16a2d1767392-0.parquet
    bulan=7/
      18c7f654841d4098812d16a2d1767392-0.parquet
    bulan=8/
      18c7f654841d4098812d16a2d1767392-0.parquet
    bulan=9/
      18c7f654841d4098812d16a2d1767392-0.parquet
  kategori=Fashion/
    bulan=1/
      18c7f654841d4098812d16a2d1767392-0.parquet
    bulan=10/
      18c7

In [42]:
# Query data terpartisi: hanya baca partisi Elektronik
print("\n=== Query pada Data Terpartisi ===")
start = time.perf_counter()
hasil_partisi = (
    pl.scan_parquet(f"{output_dir}/kategori=Elektronik/**/*.parquet")
    .filter(pl.col("status") == "selesai")
    .select([
        pl.col("total_bayar").sum().alias("total"),
        pl.col("transaksi_id").count().alias("jumlah"),
    ])
    .collect()
)
print(f"Waktu query: {time.perf_counter()-start:.4f} detik")
print(hasil_partisi)


=== Query pada Data Terpartisi ===
Waktu query: 0.1355 detik
shape: (1, 2)
┌────────────────┬────────┐
│ total          ┆ jumlah │
│ ---            ┆ ---    │
│ i64            ┆ u32    │
╞════════════════╪════════╡
│ 15771542326666 ┆ 372400 │
└────────────────┴────────┘


---
## Ringkasan Pertemuan 2

Hari ini kita telah mempelajari:
1. Arsitektur Polars: Rust + Apache Arrow = kecepatan dan efisiensi memori
2. Lazy vs Eager API: gunakan Lazy untuk data besar agar query dioptimalkan otomatis
3. Polars Expressions: cara bekerja kolumnar yang efisien
4. scan_parquet / scan_csv: lazy loading untuk file berukuran besar
5. Joins: inner, left, outer join di Polars
6. Window Functions: agregasi dalam grup tanpa mengurangi baris
7. String & Temporal operations
8. Partisi Parquet untuk query yang lebih cepat

### Pertemuan Berikutnya
DuckDB & Format Data — SQL analitik in-process, query langsung ke file Parquet,
dan interoperabilitas DuckDB <-> Polars via Apache Arrow
